In [0]:
# dbutils.library.restartPython()

In [0]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
from pyspark.sql import *
from pyspark.sql.types import  *
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from pyspark.sql.session import SparkSession
import joblib
# import plotly.express as px
# import plotly.io as pio

In [0]:
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics, feature_selection
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer

In [0]:
spark= SparkSession.builder.appName("Fraud").getOrCreate()

In [0]:
test_df= spark.read.csv("/Volumes/workspace/default/credit_fraud_project/archive/fraudTest.csv", header=True)
train_df= spark.read.csv("/Volumes/workspace/default/credit_fraud_project/archive/fraudTrain.csv", header=True)

In [0]:
train_df.display()

In [0]:
train_df.withColumn('unix_time', F.from_unixtime('unix_time')).display()

In [0]:
cols_to_drop = [
    "_c0",
    "first",
    "last",
    "street",
    "trans_num",
    "unix_time"
]

train_df = train_df.drop(*cols_to_drop)

In [0]:
train_df.groupBy('is_fraud').agg(F.count("*").alias('Total')).plot.pie('is_fraud', 'Total')

In [0]:
train_df.select(*[c for c, t in train_df.dtypes if t=="string"]).display()

In [0]:
max_date= train_df.select(
    F.max(F.to_date('trans_date_trans_time'))
    ).first()[0]

train_df= train_df.withColumns({
    'trans_date_trans_time':F.col("trans_date_trans_time").cast(TimestampType()), 
    'cc_num':F.col("cc_num").cast(LongType()), 
    'amt':F.col("amt").cast(DoubleType()), 
    'zip':F.col("zip").cast(IntegerType()), 
    'lat':F.col("lat").cast(DoubleType()),
    'long':F.col("long").cast(DoubleType()),
    'city_pop':F.col("city_pop").cast(IntegerType()), 
    'dob':F.col("dob").cast(DateType()), 
    'merch_lat':F.col("merch_lat").cast(DoubleType()),
    'merch_long':F.col("merch_long").cast(DoubleType()),
    'is_fraud':F.col("is_fraud").cast(IntegerType()),
    'hour':F.hour('trans_date_trans_time'),
    'month':F.month('trans_date_trans_time'),
    'month':F.weekofyear('trans_date_trans_time')
    })
            
train_df= train_df.withColumn(
    'current_age',
    F.round(
        F.datediff(
            F.lit(max_date)
            , F.col('dob')
        ) / 365, 0
    )
    )
train_df= train_df.drop('dob')

In [0]:
train_df.select(F.col('amt').cast(DoubleType()).alias('amt')).where(F.col("amt")<200).plot.hist()

In [0]:
merchant_sum= train_df.groupBy("merchant")\
            .agg(F.sum("amt").alias("total"))
top10_sum= merchant_sum.orderBy(F.desc("total")).limit(10).agg(F.sum("total").alias("top10_total")).collect()[0]['top10_total']

other_sum= train_df.select("amt").agg(F.sum("amt").alias("other_total")).collect()[0]['other_total']
labels= [
    f"Top 10 ($ {top10_sum:.2f})", 
    f"Other ($ {other_sum:.2f})"
]

plt.figure(figsize=(10,5))
plt.pie([ top10_sum, other_sum], labels=labels , autopct='%1.2f%%')

In [0]:
## Just to check Non Null values

# train_df.filter(
#     reduce(
#         lambda x,y : x & y,
#         [F.col(c).isNull() for c in train_df.columns]
#     )
# ).display()

In [0]:
train_df.columns

In [0]:
train_df.groupBy('merchant').agg(F.sum("amt").alias('Total Transaction Value')).orderBy("Total Transaction Value", ascending=False).limit(10).plot.bar(x='merchant', y='Total Transaction Value')

In [0]:
train_df.select(*[c for c, t in train_df.dtypes if t!="string"]).describe().display()

In [0]:
train_df.groupBy("job").agg(F.count("*").alias("count")).orderBy(F.desc('count')).limit(30).plot.bar(x="job", y='count')

In [0]:
plt.figure(figsize=(12,5))
sns.heatmap(train_df.select(*[c for c, t in train_df.dtypes if t!="string" and t!='date']).toPandas().corr(), annot=True, cmap='jet_r')
plt.title('Correlation Matrix')
plt.show()

In [0]:
train_df.where(F.col('is_fraud')==1).select('amt').plot.hist(x='amt')

In [0]:
train_df.where(F.col('is_fraud')==1).select('amt').pandas_api().plot.box()

In [0]:
train_df.withColumn('amt', F.col("amt").cast(DoubleType())).display()

In [0]:
train_df.select(
    F.count_distinct("merchant").alias("merchant_count"), 
    F.count_distinct("category").alias('category_count'),
    F.count_distinct("job").alias('job_count'),
    F.count_distinct("city").alias('city_count'),
    F.count_distinct("state").alias('state_count'),
    F.count_distinct("zip").alias('zip_count')
    ).display()

train_df.where(F.col("is_fraud")==1).select(
    F.count_distinct("merchant").alias("merchant_count"), 
    F.count_distinct("category").alias('category_count'),
    F.count_distinct("job").alias('job_count'),
    F.count_distinct("city").alias('city_count'),
    F.count_distinct("state").alias('state_count'),
    F.count_distinct("zip").alias('zip_count')
    ).display()

In [0]:
train_df.where(F.col("is_fraud")==1).groupby('gender').agg(F.sum("amt").alias('fraud_value')).plot.bar(x='gender', y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn(
    'year', F.year('trans_date_trans_time').alias('year')
    ).groupby('category', 'year').agg(
        F.sum("amt").alias('fraud_value')
        ).orderBy(F.desc('fraud_value')).pandas_api().pivot(
            index='category', columns='year', values='fraud_value'
            ).plot.bar(barmode='group')

In [0]:
train_df.where(F.col("is_fraud")==1).groupby('hour').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).plot.bar(x='hour', y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn('Days of month', F.dayofmonth('trans_date_trans_time')).groupby('Days of month').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).plot.bar(x='Days of month', y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn('Days of week', F.date_format('trans_date_trans_time', 'EEEE')).groupby('Days of week').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).pandas_api().set_index('Days of week').plot.bar(y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn('month', F.month('trans_date_trans_time')).groupby('month').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).pandas_api().plot.bar(x='month', y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn('weeks of year', F.weekofyear('trans_date_trans_time')).groupby('weeks of year').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).plot.bar(x='weeks of year', y='fraud_value')

In [0]:
train_df.where(F.col("is_fraud")==1).withColumn('year', F.year('trans_date_trans_time')).groupby('year').agg(F.sum("amt").alias('fraud_value')).orderBy(F.desc('fraud_value')).plot.bar(x='year', y='fraud_value')

In [0]:
train_df.where(F.col('is_fraud')==1).withColumn(
    'year', F.year('trans_date_trans_time')
    ).groupBy('year', 'month').agg(F.sum("amt").alias('fraud_value')).orderBy('year', 'month').pandas_api()\
        .pivot(index='month', columns='year', values='fraud_value').fillna(0).reindex([
        'January', 'February', 'March', 'April',
        'May', 'June', 'July', 'August',
        'September', 'October', 'November', 'December'
    ]).plot.bar(barmode='group')

In [0]:
(
    train_df.where(F.col('is_fraud')==1).withColumn(
        'year', F.year('trans_date_trans_time')
    ).groupBy('year', 'hour').agg(
        F.sum("amt").alias('fraud_value')
        ).orderBy('year', 'hour')
    .pandas_api().pivot(index='hour', columns='year', values='fraud_value').fillna(0)
).plot.bar(barmode='group')

In [0]:
(
    train_df.where(F.col('is_fraud')==1).withColumn(
        'year', F.year('trans_date_trans_time')
    ).groupBy('year', 'hour').agg(
        F.count('trans_date_trans_time').alias('count')
        ).orderBy('year', 'hour')
    .pandas_api().pivot(index='hour', columns='year', values='count').fillna(0)
).plot.bar(barmode='group').update_layout(
    title='Fraud Transactions Count by Hour and Year',
    xaxis_title='Hour',
    yaxis_title='Count'
)

In [0]:
# Average Fraud Amount by Hour and Year

(
    (
        train_df.where(F.col('is_fraud')==1).withColumn(
            'year': F.year('trans_date_trans_time')
        ).groupBy('year', 'hour').agg(
            F.sum("amt").alias('fraud_value')
            ).orderBy('year', 'hour')
        .pandas_api().pivot(index='hour', columns='year', values='fraud_value').fillna(0)
    ) 
    / 
    (
        train_df.where(F.col('is_fraud')==1).withColumn(
            'year': F.year('trans_date_trans_time')
        ).groupBy('year', 'hour').agg(
            F.count('trans_date_trans_time').alias('count')
            ).orderBy('year', 'hour')
        .pandas_api().pivot(index='hour', columns='year', values='count').fillna(0)
    )
).plot.line().update_layout(
    title='Average Fraud Transactions Value by Hour',
    xaxis_title='Hour',
    yaxis_title='Average Fraud Transaction'
)

In [0]:
train_df.where(F.col('is_fraud')==1).select('current_age').plot.hist(bins=5).update_layout(
    title='Fraud transaction distribution by age',
    xaxis_title= 'Age Distribution',
    yaxis_title= 'Fraud Transactions',
    width=1000
)

In [0]:
fraud_trans= train_df.filter(F.col('is_fraud')==1)
non_fraud_trans= train_df.filter(F.col('is_fraud')==0)
fraud_count= fraud_trans.count()
non_fraud_count= non_fraud_trans.count()
fraction= np.divide(fraud_count, non_fraud_count)
sampled_non_fraud= non_fraud_trans.sample(withReplacement=False, fraction=(fraction*1.5), seed=42)
balanced_df= fraud_trans.union(sampled_non_fraud)
balanced_df.groupBy('is_fraud').count().display()

In [0]:
X= balanced_df.drop('is_fraud','trans_date_trans_time').toPandas()
y= balanced_df.toPandas()['is_fraud']

In [0]:
X.head()

In [0]:
cat_cols= X.select_dtypes(include='object').columns.tolist()
num_cols= X.select_dtypes(exclude='object').columns.tolist()

tree_cat_pipeline= Pipeline([
    ('Ordinal encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

linear_cat_pipeline= Pipeline([
    ('OHE', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

num_pipeline= Pipeline([
    ('scaler', RobustScaler())
])

tree_prep= ColumnTransformer(
    [
        ('tree_cat', tree_cat_pipeline, cat_cols),
        ('tree_num', num_pipeline, num_cols)
     ], remainder='passthrough'
)

linear_prep= ColumnTransformer(
    [
        ('linear_cat', linear_cat_pipeline, cat_cols),
        ('linear_num', num_pipeline, num_cols)
     ], remainder='passthrough'
)

In [0]:
X_train, X_test, y_train, y_test= train_test_split(X,y,train_size=0.8,random_state=42,stratify=y)
kf= KFold(n_splits=5, shuffle=True, random_state=42)

In [0]:
X_train_linear= linear_prep.fit_transform(X_train)
X_test_linear= linear_prep.transform(X_test)
X_train_tree= tree_prep.fit_transform(X_train)
X_test_tree= tree_prep.transform(X_test)

1. RidgeClassifier
2. LogisticRegression 
3. ExtraTreesClassifier
4. RandomForestClassifier 
5. GradientBoostingClassifier
6. DecisionTreeClassifier

In [0]:
linear_pipe=Pipeline([
    ('model',RidgeClassifier())
])

tree_pipe=Pipeline([
    ('model', ExtraTreesClassifier())
])
linear_grid= [

    {
        'model':[RidgeClassifier()],
        'model__alpha':np.linspace(0.01,10,10),
        'model__max_iter':[100,200,300],
        'model__random_state':[42]
    },
    {
        'model':[LogisticRegression()],
        'model__penalty':['l2'],
        'model__max_iter':[5000],
        'model__random_state':[42]
    }
]

tree_grid= [
    {
        'model':[ExtraTreesClassifier()],
        'model__max_depth':[10,15,20,None],
        'model__max_features':['sqrt','log2',None],
        'model__ccp_alpha':[1e-4,1e-2,0.2,0.5],
        'model__n_jobs':[-1],
        'model__random_state':[42]
    },
    {
        'model':[RandomForestClassifier()],
        'model__max_depth':[10,15,20,None],
        'model__max_features':['sqrt','log2',None],
        'model__ccp_alpha':[1e-4,1e-2,0.2,0.5],
        'model__n_jobs':[-1],
        'model__random_state':[42]
    },
    {
        'model':[GradientBoostingClassifier()],
        'model__loss':['log_loss','exponential'],
        'model__learning_rate':np.linspace(0.01,1,5),
        'model__n_estimators':[100,200,300],
        'model__random_state':[42]
    },
    {
        'model':[DecisionTreeClassifier()],
        'model__max_depth':[10,15,20,None],
        'model__max_features':['sqrt','log2',None],
        'model__random_state':[42],
        'model__ccp_alpha':[1e-4,1e-2,0.2]
    }
]

# linear_search= GridSearchCV(linear_pipe, param_grid=linear_grid, cv=kf, scoring='f1', n_jobs=-1)
# linear_search.fit(X_train_linear,y_train)
# linear_estimator= linear_search.best_estimator_
# print(linear_search.best_params_)

tree_search= GridSearchCV(tree_pipe, param_grid=tree_grid, cv=kf, scoring='f1', n_jobs=-1)
tree_search.fit(X_train_tree,y_train)
tree_estimator= tree_search.best_estimator_
print(tree_search.best_params_)

In [0]:
# linear_search = joblib.load('linear_estimator')
tree_search = joblib.load('tree_estimator')

In [0]:
linear_features= linear_prep.get_feature_names_out()
linear_model= linear_estimator.best_estimator_.named_steps['model']
linear_coef= linear_model.coef_[0]
linear_df= pd.DataFrame({
    'features':linear_features,
    'coef':linear_coef,
    'abs_coef':np.abs(linear_coef)
})

def get_group(feature_names):
    cleaned= feature_names.split("__")[-1]

    if "_" in cleaned:
        return cleaned.split("_")[0]
    return cleaned

linear_df['group']= linear_df['features'].apply(get_group)
linear_df['importance_norm']= np.divide(linear_df['abs_coef'],linear_df['abs_coef'].sum())
linear_df= linear_df.sort_values(by='importance_norm', ascending=False)

In [0]:
linear_df.groupby('group')['importance_norm'].sum().plot(kind='bar', x='group', y='importance_norm')

In [0]:
tree_features= tree_prep.get_feature_names_out() if hasattr(X_train_tree, 'get_feature_names_out') else 'Yo'
tree_importances= tree_estimator.named_steps['model'].feature_importances_

In [0]:
tree_features

In [0]:
# Get tree feature importances
tree_features = tree_prep.get_feature_names_out()
tree_importances = tree_estimator.named_steps['model'].feature_importances_

# Create DataFrame
tree_df = pd.DataFrame({
    'features': tree_features,
    'importance': tree_importances,
})

# Extract feature groups
def get_group(feature_names):
    cleaned = feature_names.split("__")[-1]
    if "_" in cleaned:
        return cleaned.split("_")[0]
    return cleaned

tree_df['group'] = tree_df['features'].apply(get_group)
tree_df['importance_norm'] = np.divide(tree_df['importance'], tree_df['importance'].sum())
tree_df = tree_df.sort_values(by='importance_norm', ascending=False)

display(tree_df.head(10))

In [0]:
# Visualize grouped feature importances for tree model
tree_df.groupby('group')['importance'].sum().sort_values(ascending=False).plot(
    kind='bar', 
    title='Feature Importance by Group (Tree Model)',
    xlabel='Feature Group',
    ylabel='Normalized Importance'
)

In [0]:
# joblib.dump(linear_estimator, 'linear_estimator')
# joblib.dump(tree_estimator, 'tree_estimator')

In [0]:
cm= metrics.confusion_matrix(y_test, tree_search.predict(X_test_tree))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
linear_search.best_estimator_.fit(X_train_linear, y_train)  # Fit the model if not already fittedcm= metrics.confusion_matrix(y_test, linear_search.best_estimator_.predict(X_test_linear))

cm= metrics.confusion_matrix(y_test, linear_search.best_estimator_.predict(X_test_linear))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
print(linear_search.best_score_)
print(tree_search.best_score_)